## Example 1 — the version, and a WAV written by soundfile comes back through librosa.load

In [ ]:
import importlib.metadata, os, tempfile
import numpy as np
import soundfile as sf
import librosa

assert librosa.__version__ == importlib.metadata.version("librosa"), librosa.__version__
sr = 22050
tone = 0.5 * np.sin(2 * np.pi * 440 * np.arange(sr) / sr)
with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, "tone.wav")
    sf.write(path, tone, sr, subtype="PCM_16")
    y, got_sr = librosa.load(path, sr=None)
assert got_sr == sr and y.dtype == np.float32 and len(y) == sr, (got_sr, y.dtype, len(y))
assert abs(librosa.get_duration(y=y, sr=sr) - 1.0) < 1e-9
assert np.max(np.abs(y - tone)) < 1e-3, np.max(np.abs(y - tone))
print("librosa", librosa.__version__, "read", len(y), "samples at", got_sr, "Hz")

## Example 2 — resampling 44.1 kHz to 16 kHz through soxr keeps a 1 kHz tone at 1 kHz

In [ ]:
import numpy as np
import librosa

y = np.sin(2 * np.pi * 1000 * np.arange(44100) / 44100).astype(np.float32)
for res_type in ("soxr_hq", "soxr_vhq", "soxr_qq"):
    z = librosa.resample(y, orig_sr=44100, target_sr=16000, res_type=res_type)
    assert len(z) == 16000, (res_type, len(z))
    spectrum = np.abs(np.fft.rfft(z))
    peak_hz = np.argmax(spectrum) * 16000 / len(z)
    assert abs(peak_hz - 1000) <= 1, (res_type, peak_hz)
    print(res_type, "->", len(z), "samples, peak at", peak_hz, "Hz")

## Example 3 — spectrogram, mel features, MFCCs and the zero-crossing rate (numba-compiled)

In [ ]:
import numpy as np
import librosa

sr = 16000
y = np.sin(2 * np.pi * 440 * np.arange(sr) / sr).astype(np.float32)
S = np.abs(librosa.stft(y, n_fft=1024, hop_length=256))
freqs = librosa.fft_frequencies(sr=sr, n_fft=1024)
assert abs(freqs[S.mean(axis=1).argmax()] - 440) < sr / 1024, freqs[S.mean(axis=1).argmax()]
mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64)
mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
assert mel.shape[0] == 64 and mfcc.shape[0] == 13 and mel.shape[1] == mfcc.shape[1], (mel.shape, mfcc.shape)
zcr = librosa.feature.zero_crossing_rate(y)[0]
# A 440 Hz sine crosses zero 880 times a second.
assert abs(np.median(zcr) * sr - 880) < 40, np.median(zcr) * sr
print("stft", S.shape, "mel", mel.shape, "mfcc", mfcc.shape, "zcr/s", round(float(np.median(zcr) * sr)))

## Example 4 — pitch: the YIN tracker finds A4, and note names round-trip

In [ ]:
import numpy as np
import librosa

sr = 16000
y = np.sin(2 * np.pi * 440 * np.arange(sr) / sr).astype(np.float32)
f0 = librosa.yin(y, fmin=65, fmax=2093, sr=sr)
assert abs(np.median(f0) - 440) < 2, np.median(f0)
assert librosa.hz_to_note(440.0) == "A4" and librosa.note_to_hz("A4") == 440.0
assert librosa.midi_to_note(60) == "C4" and librosa.note_to_midi("C4") == 60
print("median f0", round(float(np.median(f0)), 2), "Hz =", librosa.hz_to_note(float(np.median(f0))))

## Example 5 — beat tracking on a 120 BPM click track, and a missing file raises an error

In [ ]:
import numpy as np
import soundfile
import librosa

sr = 22050
times = np.arange(0, 20, 0.5)  # a click every half second = 120 beats per minute
y = librosa.clicks(times=times, sr=sr, length=sr * 20)
tempo, beats = librosa.beat.beat_track(y=y, sr=sr)
tempo = float(np.atleast_1d(tempo)[0])
# The tempo estimate moves on a grid (60 * sr / (512 * lag)): 117.45 and 123.05 are the two values next to 120.
assert abs(tempo - 120) < 3, tempo
assert len(beats) > 30, len(beats)
try:
    librosa.load("/nonexistent/file.wav")
    raise AssertionError("a missing file loaded")
except (FileNotFoundError, soundfile.LibsndfileError) as error:  # librosa 1.0 reads only through soundfile
    print("refused as expected:", type(error).__name__)
print("tempo", round(tempo, 1), "BPM,", len(beats), "beats")